In [ ]:
# Model 1
from gurobipy import Model, GRB, quicksum
import pandas as pd

# -------------- Data Preparation --------------
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')

N = 3
alpha_init = new_df['alpha'].head(N).tolist()
print("csemCur number:", new_df['csemCur'].head(N).tolist())
print("alpha_init:", alpha_init)
V = new_df['Volume'].head(N).tolist() 
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
print("K:", K)

T_years = 7
Y = 365
Delta_P = [new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i] 
           for i in range(N)]
Delta_P = [round(float(dp), 3) for dp in Delta_P]  # delta p for csems with different nominal pressure
print("Delta_P:", Delta_P)

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]

mu_COPER = 1.6
mu_MANT  = 4.2
M = max(alpha_init)

mdl = Model("yearly_gas_maintenance_LEAK_FIXED")
mdl.Params.OutputFlag = 1

# -------------- Decision Variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, name="alpha")

# -------------- Constraints --------------

# 1. Initial alpha
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i], name=f"alpha_init_{i}")

# 2. At most one maintenance per year/point
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1, name=f"one_action_{i}_{t}")

# 3. alpha evolution
for i in range(N):
    for t in range(1, T_years+1):
        # No action
        growth = 1.01 * alpha[i,t-1]
        mdl.addConstr(alpha[i,t] >= growth - M * (x_COPER[i,t-1] + x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] <= growth + M * (x_COPER[i,t-1] + x_MANT[i,t-1]))
        
        # COPER
        maint_coper = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] >= maint_coper - M * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] <= maint_coper + M * (1 - x_COPER[i,t-1]))
        
        # MANT
        maint_mant = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] >= maint_mant - M * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] <= maint_mant + M * (1 - x_MANT[i,t-1]))

# 🔥 NEW: 泄漏约束 - 防止连续每年行动 (最简最稳!)
for i in range(N):
    for t in range(1, T_years):  # t=1..6，检查年t-1和t
        mdl.addConstr(
            x_COPER[i,t-1] + x_MANT[i,t-1] + x_COPER[i,t] + x_MANT[i,t] <= 1,
            name=f"no_consecutive_{i}_{t}"
        )

# -------------- Objective (不变) --------------
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P[i]) * K[i] * C_leak_t[t]  
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# -------------- Solve Parameters --------------
mdl.Params.MIPGap = 0.05
mdl.Params.IntFeasTol = 1e-9
mdl.Params.FeasibilityTol = 1e-9  
mdl.Params.OptimalityTol = 1e-9
mdl.Params.NumericFocus = 3
mdl.Params.ScaleFlag = 2

# -------------- Solve & Output --------------
mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ 求解成功! Total cost: {mdl.objVal:.2f}")
    
    print("\n=== 维护计划 ===")
    total_actions = 0
    for i in range(N):
        print(f"\nCSEM {i}:")
        for t in range(T_years):
            if x_COPER[i,t].X > 0.5:
                print(f"  第{t+1}年: COPER (α: {alpha[i,t].X:.6f} → {alpha[i,t+1].X:.6f})")
                total_actions += 1
            elif x_MANT[i,t].X > 0.5:
                print(f"  第{t+1}年: MANT (α: {alpha[i,t].X:.6f} → {alpha[i,t+1].X:.6f})")
                total_actions += 1
            else:
                print(f"  第{t+1}年: 无维护 (α={alpha[i,t+1].X:.6f})")
    
    print(f"\n📊 总行动次数: {total_actions} (原模型会更多！)")
    
    # 验证泄漏量（可选）
    print("\n=== 各年泄漏量验证 (kg) ===")
    for i in range(N):
        leaks = []
        for t in range(T_years):
            leak_t = (Y * alpha[i,t].X / Delta_P[i]) * K[i]
            action_t = "★" if x_COPER[i,t].X + x_MANT[i,t].X > 0.5 else "-"
            leaks.append(f"Y{t+1}:{leak_t:.1f}{action_t}")
        print(f"CSEM {i}: [{', '.join(leaks)}]")
        
else:
    print(f"❌ Status: {mdl.status}")
    if mdl.status == GRB.INFEASIBLE:
        mdl.computeIIS()
        print("冲突约束:")
        for c in mdl.getConstrs():
            if c.IISConstr:
                print(f"  {c.ConstrName}")


csemCur number: [5000200795, 5000252448, 5000252449]
alpha_init: [0.0049148948437029, 0.1816128661238826, 0.022172033787185]
K: [45.42857142857141, 378.5714285714285, 377.71428571428595]
Delta_P: [0.301, 0.301, 0.301]
Set parameter Username
Set parameter LicenseID to value 2773682
Academic license - for non-commercial use only - expires 2027-02-02

🛡️ 添加泄漏约束：禁止连续行动...
Set parameter MIPGap to value 0.05
Set parameter IntFeasTol to value 1e-09
Set parameter FeasibilityTol to value 1e-09
Set parameter OptimalityTol to value 1e-09
Set parameter NumericFocus to value 3
Set parameter ScaleFlag to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 168 rows, 66 columns and 537 nonzeros
Model fingerprint: 0xfbe14e17
Variable types: 24 continuous, 42 integer (42 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+00]
  Objective range  [2e+04

In [3]:
from gurobipy import Model, GRB, quicksum
import pandas as pd

# -------------- Data (相同) --------------
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')
N = 3
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
T_years = 7
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
C_COPER, C_MANT = 15600, 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER, mu_MANT = 1.6, 4.2
M_alpha = max(alpha_init)

# 🔥 计算第一年泄漏，决定是否允许第一年行动
first_year_leaks = [(Y * alpha_init[i] / Delta_P[i] * K[i]) for i in range(N)]
print("第一年泄漏量 (kg):", first_year_leaks)

mdl = Model("leak_based_corrected")
mdl.Params.OutputFlag = 1

# -------------- 变量 --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY)
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY)
alpha   = mdl.addVars(N, T_years+1, lb=0)
L       = mdl.addVars(N, T_years, lb=0)  # 累计泄漏

# -------------- 约束 --------------
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i])

    for t in range(T_years):
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= 1)

    for t in range(1, T_years+1):
        # alpha演化（原样）
        growth = 1.01 * alpha[i,t-1]
        x_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(alpha[i,t] >= growth - M_alpha * x_prev)
        mdl.addConstr(alpha[i,t] <= growth + M_alpha * x_prev)
        mdl.addConstr(alpha[i,t] >= alpha[i,t-1]/mu_COPER - M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] <= alpha[i,t-1]/mu_COPER + M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] >= alpha[i,t-1]/mu_MANT - M_alpha * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] <= alpha[i,t-1]/mu_MANT + M_alpha * (1 - x_MANT[i,t-1]))

# 🔥 累计泄漏演化（修正版）
M_leak = 1e6
for i in range(N):
    # 第一年特殊处理：如果第一年泄漏>2kg，允许行动；否则强制无行动
    if first_year_leaks[i] > 2:
        mdl.addConstr(L[i,0] == 0)  # 形式上0，但不强制无行动
        print(f"CSEM{i}: 第一年{first_year_leaks[i]:.1f}kg > 2kg，允许行动")
    else:
        mdl.addConstr(x_COPER[i,0] + x_MANT[i,0] == 0)
        mdl.addConstr(L[i,0] == 0)
        print(f"CSEM{i}: 第一年{first_year_leaks[i]:.1f}kg <= 2kg，强制无行动")

    # 从第二年开始累计泄漏
    for t in range(1, T_years):
        leak_prev = (Y * alpha[i,t-1] / Delta_P[i]) * K[i]
        x_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        
        # L[t] = L[t-1] + leak_prev（无行动）or 0（有行动）
        mdl.addConstr(L[i,t] >= L[i,t-1] + leak_prev - M_leak * x_prev)
        mdl.addConstr(L[i,t] <= L[i,t-1] + leak_prev + M_leak * x_prev)
        mdl.addConstr(L[i,t] <= M_leak * x_prev)

# 🔥 泄漏触发约束：只有 L[i,t] > 2kg 才允许第t年行动
LEAK_THRESHOLD = 2.0
for i in range(N):
    for t in range(T_years):
        # 第一年：如果泄漏大，直接允许；否则已强制无行动
        if t == 0 and first_year_leaks[i] > 2:
            continue  # 第一年自由决策
            
        # 其他年：严格按累计泄漏
        mdl.addConstr(x_COPER[i,t] + x_MANT[i,t] <= L[i,t] / LEAK_THRESHOLD,
                     name=f"leak_trigger_{i}_{t}")

# -------------- 目标函数 --------------
leak_cost = quicksum((Y * alpha[i,t] / Delta_P[i]) * K[i] * C_leak_t[t] 
                     for i in range(N) for t in range(T_years))
action_cost = quicksum(C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t] 
                       for i in range(N) for t in range(T_years))
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

mdl.Params.MIPGap = 0.05
mdl.Params.IntFeasTol = 1e-9
mdl.Params.FeasibilityTol = 1e-9
mdl.Params.OptimalityTol = 1e-9
mdl.Params.NumericFocus = 3
mdl.Params.ScaleFlag = 2

mdl.optimize()

if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total cost: {mdl.objVal:.2f}")
    print("\n=== 维护计划 (严格按泄漏触发) ===")
    for i in range(N):
        print(f"\nCSEM {i} (初始泄漏{first_year_leaks[i]:.1f}kg):")
        for t in range(T_years):
            act = "COPER" if x_COPER[i,t].X > 0.5 else "MANT" if x_MANT[i,t].X > 0.5 else "-"
            print(f"  Y{t+1}: {act} (L={L[i,t].X:.1f}kg)")
else:
    print(f"❌ Status: {mdl.status}")
    if mdl.status == GRB.INFEASIBLE:
        mdl.computeIIS()
        print("冲突约束:")
        for c in mdl.getConstrs():
            if c.IISConstr:
                print(f"  {c.ConstrName}")


第一年泄漏量 (kg): [270.7507567672498, 83372.11426593518, 10155.359718396188]
CSEM0: 第一年270.8kg > 2kg，允许行动
CSEM1: 第一年83372.1kg > 2kg，允许行动
CSEM2: 第一年10155.4kg > 2kg，允许行动
Set parameter MIPGap to value 0.05
Set parameter IntFeasTol to value 1e-09
Set parameter FeasibilityTol to value 1e-09
Set parameter OptimalityTol to value 1e-09
Set parameter NumericFocus to value 3
Set parameter ScaleFlag to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 225 rows, 87 columns and 756 nonzeros
Model fingerprint: 0xd0ccb73b
Variable types: 45 continuous, 42 integer (42 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [2e+04, 1e+09]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 1e+00]
Presolve removed 198 rows and 73 columns
Presolve time: 0.01s
Presolved: 27 rows, 14 columns, 82 nonzeros
Variable types: 6 continu

In [ ]:
from gurobipy import Model, GRB, quicksum
import pandas as pd

# -------------- 数据准备 --------------
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')
N = 3
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
T_years = 7
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 2

first_year_leaks = [Y * alpha_init[i] / Delta_P[i] * K[i] for i in range(N)]
print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- 决策变量 --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, name="alpha")
last_action = mdl.addVars(N, T_years+1, vtype=GRB.INTEGER, lb=-1, ub=T_years, name="last_action")

# 初始化
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i])
    mdl.addConstr(last_action[i,0] == -1)

# 🔥 修复的alpha演化：精确维护效果
M_alpha = max(alpha_init) * 10  # 增大Big-M
for i in range(N):
    for t in range(1, T_years+1):
        # 无维护：自然增长1.01倍
        growth = 1.01 * alpha[i,t-1]
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(alpha[i,t] >= growth - M_alpha * (1 - no_action))
        mdl.addConstr(alpha[i,t] <= growth + M_alpha * (1 - no_action))
        
        # COPER维护：精确到 alpha[t-1]/1.6
        coper_target = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] <= coper_target + M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] >= coper_target - M_alpha * (1 - x_COPER[i,t-1]))
        
        # MANT维护：精确到 alpha[t-1]/4.2
        mant_target = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] <= mant_target + M_alpha * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] >= mant_target - M_alpha * (1 - x_MANT[i,t-1]))



# last_action更新
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # 有行动时 = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # 无行动时继承
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# 🔥 核心：累积泄漏约束（完全线性化）
LEAK_TH = 2.0
BIG_M_LEAK = 1e6

for i in range(N):
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # 每年最多一次
        
        last_t = last_action[i,t]
        
        # 简化的二元决策：基于年份间隔的泄漏近似
        years_since_last = t - last_action[i,t]  # 线性表达式
        leak_proxy = first_year_leaks[i] * years_since_last  # 近似累积泄漏
        
        # 约束：要做行动，必须累积泄漏>=2kg
        mdl.addConstr(leak_proxy >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year))
        
        # 防止负值
        mdl.addConstr(leak_proxy >= 0)

# 目标函数
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P[i]) * (Delta_P[i]) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# 求解参数
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()

# if mdl.status == GRB.OPTIMAL:
#     print(f"\n✅ 总成本: {mdl.objVal:.0f}")
#     print("\n=== 严格累积泄漏决策计划 ===")
#     for i in range(N):
#         print(f"\nCSEM {i+1} (首年泄漏{first_year_leaks[i]:.0f}kg):")
#         for t in range(T_years):
#             act = "COPER" if x_COPER[i,t].X > 0.5 else ("MANT" if x_MANT[i,t].X > 0.5 else "-")
#             last_t_val = last_action[i,t].X
#             gap_years = t - max(last_t_val, -1)
#             leak_proxy_val = first_year_leaks[i] * gap_years
#             status = "✅满足" if leak_proxy_val >= 2 else "❌不足"
#             print(f"  Y{t+1}: {act:<6} | 上次Y{max(last_t_val+1,0)} | "
#                   f"间隔{gap_years}年 | 预估泄漏{leak_proxy_val:.1f}kg | {status}")
                  
# elif mdl.status == GRB.INFEASIBLE:
#     print("\n❌ 无解")
#     mdl.computeIIS()
#     mdl.write("infeasible.ilp")
# else:
#     print(f"\n状态: {mdl.status} (目标值: {mdl.objVal if hasattr(mdl, 'objVal') else 'N/A'})")




# 在模型优化后添加这个输出代码（替换原来的输出部分）
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ 总成本: {mdl.objVal:,.0f}")
    
    print("\n=== 详细维护计划与Alpha演化 ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        print(f"\n{'='*60}")
        print(f"CSEM {i+1} (初始α={alpha_init[i]:.6f}, 首年泄漏{first_year_leaks[i]:.0f}kg)")
        print(f"{'='*60}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            alpha_start = alpha[i,t].X
            alpha_end = alpha[i,t+1].X
            leak_kg = Y * alpha_start / Delta_P[i] * K[i]
            leak_cost_t = leak_kg * C_leak_t[t]
            csem_leak_cost += leak_cost_t
            
            act_type = ""
            act_cost = 0
            if x_COPER[i,t].X > 0.5:
                act_type = "COPER"
                act_cost = C_COPER
                csem_action_cost += act_cost
            elif x_MANT[i,t].X > 0.5:
                act_type = "MANT" 
                act_cost = C_MANT
                csem_action_cost += act_cost
            else:
                act_type = "No action"
            
            total_leak_cost += leak_cost_t
            if act_cost > 0:
                total_action_cost += act_cost
                
            alpha_reduction = (alpha_start - alpha_end) / alpha_start * 100 if alpha_start > 0 else 0
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"α: {alpha_start:.6f}→{alpha_end:.6f} "
                  f"(-{alpha_reduction:5.1f}%) | "
                  f"泄漏{leak_kg:>7.0f}kg | "
                  f"成本{leak_cost_t:>9,.0f}")
        
        print(f"  CSEM{i+1} 小计: 行动{csem_action_cost:>9,.0f} + 泄漏{csem_leak_cost:>10,.0f} = {csem_action_cost+csem_leak_cost:>11,.0f}")
    
    print(f"\n{'='*60}")
    print(f"🎯 总计: 行动成本 {total_action_cost:>12,.0f} + 泄漏成本 {total_leak_cost:>12,.0f} = 总成本 {mdl.objVal:>12,.0f}")
    print(f"{'='*60}")
    
    # 验证约束：检查后续年泄漏是否真的低
    print("\n🔍 验证：维护后泄漏率变化")
    for i in range(N):
        print(f"CSEM{i+1}: Y1泄漏 {first_year_leaks[i]:.0f}kg → "
              f"Y2泄漏 {Y * alpha[i,1].X / Delta_P[i] * K[i]:.0f}kg "
              f"(下降 {(1 - alpha[i,1].X/alpha[i,0].X)*100:.1f}%)")
              
else:
    print(f"❌ 求解状态: {mdl.status}")



第一年泄漏量(kg): ['270.8', '83372.1', '10155.4']
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 279 rows, 90 columns and 846 nonzeros
Model fingerprint: 0x7b50373d
Variable types: 24 continuous, 66 integer (42 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [2e+04, 1e+09]
  Bounds range     [1e+00, 7e+00]
  RHS range        [5e-03, 2e+06]
Found heuristic solution: objective 1.830554e+09
Presolve removed 205 rows and 63 columns
Presolve time: 0.01s
Presolved: 74 rows, 27 columns, 252 nonzeros
Found heuristic solution: objective 4.654648e+08
Variable types: 7 continuous, 20 integer (13 binary)

Root relaxation: objective 2.977340e+08, 9 iterations, 0.00 seconds (0.00 wo

In [25]:
from gurobipy import Model, GRB, quicksum
import pandas as pd

# -------------- 数据准备 --------------
new_df = pd.read_csv('new_df_with_volume.csv', sep=',')
N = 20
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 10

first_year_leaks = [Y * alpha_init[i] / Delta_P[i] * K[i] for i in range(N)]
print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- 决策变量 --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, name="alpha")
last_action = mdl.addVars(N, T_years+1, vtype=GRB.INTEGER, lb=-1, ub=T_years, name="last_action")

# 🔥 新增：精确累积泄漏追踪变量
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")

# 初始化
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i])
    mdl.addConstr(last_action[i,0] == -1)

# alpha演化：精确维护效果（保持原样）
for i in range(N):
    for t in range(1, T_years+1):
        # 无维护：自然增长1.01倍
        growth = 1.01 * alpha[i,t-1]
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(alpha[i,t] >= growth - M_alpha * (1 - no_action))
        mdl.addConstr(alpha[i,t] <= growth + M_alpha * (1 - no_action))
        
        # COPER维护：精确到 alpha[t-1]/1.6
        coper_target = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] <= coper_target + M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] >= coper_target - M_alpha * (1 - x_COPER[i,t-1]))
        
        # MANT维护：精确到 alpha[t-1]/4.2
        mant_target = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] <= mant_target + M_alpha * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] >= mant_target - M_alpha * (1 - x_MANT[i,t-1]))

# last_action更新（保持原样）
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # 有行动时 = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # 无行动时继承
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# 🔥 新增：年泄漏量定义
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (Y * alpha[i,t] / Delta_P[i]) * Delta_P[i] * K[i])

# 🔥 新增：精确累积泄漏递推逻辑（核心修复）
LEAK_TH = 10.0
for i in range(N):
    # 第0年：cum_leak初始化为第0年泄漏
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # 第1年到第6年：递推累积
    for t in range(1, T_years):
        # 情况1：上年无行动 → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # 情况2：上年有行动 → cum_leak[t] = yearly_leak[t]（重置）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # 🔥 核心约束：每年行动必须满足累积泄漏>=2kg
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # 每年最多一次
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year))

# 目标函数（保持原样）
leak_cost = quicksum(
    (Y * alpha[i,t] / Delta_P[i]) * (Delta_P[i]) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# 求解参数
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()

# 输出（增强版，显示cum_leak验证）
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ 总成本: {mdl.objVal:,.0f}")
    
    print("\n=== 详细维护计划与累积泄漏验证 ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (初始α={alpha_init[i]:.6f}, 首年泄漏{first_year_leaks[i]:.0f}kg)")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            alpha_start = alpha[i,t].X
            alpha_end = alpha[i,t+1].X
            leak_kg = yearly_leak[i,t].X  # 使用精确的yearly_leak
            leak_cost_t = leak_kg * C_leak_t[t]
            csem_leak_cost += leak_cost_t
            
            act_type = ""
            act_cost = 0
            if x_COPER[i,t].X > 0.5:
                act_type = "COPER"
                act_cost = C_COPER
                csem_action_cost += act_cost
            elif x_MANT[i,t].X > 0.5:
                act_type = "MANT" 
                act_cost = C_MANT
                csem_action_cost += act_cost
            else:
                act_type = "No action"
            
            total_leak_cost += leak_cost_t
            if act_cost > 0:
                total_action_cost += act_cost
                
            alpha_reduction = (alpha_start - alpha_end) / alpha_start * 100 if alpha_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "✅满足" if (cum_leak_val >= 2) else "❌违规"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"α: {alpha_start:.6f}→{alpha_end:.6f} "
                  f"(-{alpha_reduction:5.1f}%) | "
                  f"泄漏{leak_kg:>6.3f}kg | "
                  f"累积{cum_leak_val:>6.3f}kg | {valid_check}")
        
        print(f"  CSEM{i+1} 小计: 行动{csem_action_cost:>9,.0f} + 泄漏{csem_leak_cost:>10,.0f} = {csem_action_cost+csem_leak_cost:>11,.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 总计: 行动成本 {total_action_cost:>12,.0f} + 泄漏成本 {total_leak_cost:>12,.0f} = 总成本 {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")
    
else:
    print(f"❌ 求解状态: {mdl.status}")


第一年泄漏量(kg): ['270.8', '83372.1', '10155.4', '13288.9', '1058.5', '8422.0', '66236.0', '54168.6', '171702.1', '51863.7', '397.7', '6971.3', '47079.0', '124.3', '4610.9', '216.0', '996.8', '1096.9', '43938.2', '10414.8']
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 5760 rows, 2080 columns and 19780 nonzeros
Model fingerprint: 0xe2aee041
Variable types: 1040 continuous, 1040 integer (680 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+06]
  Objective range  [2e+04, 9e+08]
  Bounds range     [1e+00, 2e+01]
  RHS range        [4e-03, 1e+06]
Presolve removed 2171 rows and 918 columns
Presolve time: 0.27s
Presolved: 3589 rows, 1162 columns, 13325 nonzeros
Variable types: 462 continuous, 700